In [0]:
dbutils.widgets.text("environment", "dev")
dbutils.widgets.dropdown(
    "dataset_name",
    "products",
    [
        "products",
        "customers",
        "stores",
        "inventory",
        "promotions"
    ]
)
dbutils.widgets.text("load_date", "")
dbutils.widgets.text("pipeline_run_id", "")
dbutils.widgets.text("source_system", "retail_master_data")
dbutils.widgets.text("catalog_name", "retail_lakehouse")

In [0]:
environment = dbutils.widgets.get("environment").strip()
dataset_name = dbutils.widgets.get("dataset_name").strip().lower()
load_date = dbutils.widgets.get("load_date").strip()
pipeline_run_id = dbutils.widgets.get("pipeline_run_id").strip()
source_system = dbutils.widgets.get("source_system").strip()
catalog_name = dbutils.widgets.get("catalog_name").strip()

In [0]:
import uuid

if not pipeline_run_id:
    pipeline_run_id = str(uuid.uuid4())

print(f"Environment     : {environment}")
print(f"Dataset         : {dataset_name}")
print(f"Load date       : {load_date or 'Not required'}")
print(f"Pipeline run ID : {pipeline_run_id}")
print(f"Source system   : {source_system}")
print(f"Catalog         : {catalog_name}")

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

In [0]:
products_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("standard_price", StringType(), True),
    StructField("cost_price", StringType(), True),
    StructField("product_status", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])

In [0]:
customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("age", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("region", StringType(), True),
    StructField("loyalty_tier", StringType(), True),
    StructField("registration_date", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])

In [0]:
stores_schema = StructType([
    StructField("store_id", StringType(), True),
    StructField("store_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("region", StringType(), True),
    StructField("store_size", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])

In [0]:
inventory_schema = StructType([
    StructField("store_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("inventory_date", StringType(), True),
    StructField("opening_stock", StringType(), True),
    StructField("received_quantity", StringType(), True),
    StructField("sold_quantity", StringType(), True),
    StructField("damaged_quantity", StringType(), True),
    StructField("closing_stock", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])

In [0]:
promotions_schema = StructType([
    StructField("promotion_id", StringType(), True),
    StructField("promotion_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("discount_type", StringType(), True),
    StructField("discount_value", StringType(), True),
    StructField("start_date", StringType(), True),
    StructField("end_date", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])

In [0]:
schema_map = {
    "products": products_schema,
    "customers": customers_schema,
    "stores": stores_schema,
    "inventory": inventory_schema,
    "promotions": promotions_schema
}

In [0]:
dataset_config = {
    "products": {
        "source_folder": "products",
        "target_table": "products",
        "quarantine_table": "products_malformed_records",
        "requires_load_date": False,
        "source_system": "product_master"
    },
    "customers": {
        "source_folder": "customers",
        "target_table": "customers",
        "quarantine_table": "customers_malformed_records",
        "requires_load_date": False,
        "source_system": "customer_master"
    },
    "stores": {
        "source_folder": "stores",
        "target_table": "stores",
        "quarantine_table": "stores_malformed_records",
        "requires_load_date": False,
        "source_system": "store_master"
    },
    "inventory": {
        "source_folder": "inventory",
        "target_table": "inventory",
        "quarantine_table": "inventory_malformed_records",
        "requires_load_date": True,
        "source_system": "inventory_management"
    },
    "promotions": {
        "source_folder": "promotions",
        "target_table": "promotions",
        "quarantine_table": "promotions_malformed_records",
        "requires_load_date": False,
        "source_system": "promotion_management"
    }
}

In [0]:
if dataset_name not in dataset_config:
    raise ValueError(
        f"Unsupported dataset_name: {dataset_name}. "
        f"Allowed values: {list(dataset_config.keys())}"
    )

config = dataset_config[dataset_name]
raw_schema = schema_map[dataset_name]
source_system = config["source_system"]

In [0]:
if config["requires_load_date"] and not load_date:
    raise ValueError(
        f"load_date is required for dataset={dataset_name}. "
        "Example: 2026-07-15"
    )

In [0]:
landing_base_path = (
    f"/Volumes/{catalog_name}/landing/source_files"
)

In [0]:
if config["requires_load_date"]:
    source_path = (
        f"{landing_base_path}/"
        f"{config['source_folder']}/"
        f"load_date={load_date}"
    )
else:
    source_path = (
        f"{landing_base_path}/"
        f"{config['source_folder']}"
    )

In [0]:
bronze_table = (
    f"{catalog_name}.bronze."
    f"{config['target_table']}"
)

quarantine_table = (
    f"{catalog_name}.quarantine."
    f"{config['quarantine_table']}"
)

audit_table = (
    f"{catalog_name}.audit.pipeline_runs"
)

processed_files_table = (
    f"{catalog_name}.audit.processed_files"
)

In [0]:
print(f"Dataset               : {dataset_name}")
print(f"Source path           : {source_path}")
print(f"Bronze table          : {bronze_table}")
print(f"Quarantine table      : {quarantine_table}")
print(f"Processed-files table : {processed_files_table}")

In [0]:
try:
    path_contents = dbutils.fs.ls(source_path)

except Exception as exc:
    raise FileNotFoundError(
        f"Source path does not exist: {source_path}. "
        f"Verify that source data was generated for "
        f"dataset={dataset_name}."
    ) from exc

In [0]:
source_files = [
    file
    for file in path_contents
    if file.name.lower().endswith(".csv")
]

if not source_files:
    raise FileNotFoundError(
        f"No CSV files found inside: {source_path}"
    )

In [0]:
for file in source_files:
    print(
        f"Name: {file.name}, "
        f"Size: {file.size}, "
        f"Modified: {file.modificationTime}"
    )

In [0]:
available_file_paths = [
    file.path
    for file in source_files
]

In [0]:
from pyspark.sql import functions as F

processed_file_paths = {
    row["source_file_path"]
    for row in (
        spark.table(processed_files_table)
        .filter(
            (F.col("target_table") == bronze_table)
            & (F.col("processing_status") == "SUCCESS")
        )
        .select("source_file_path")
        .distinct()
        .collect()
    )
}

In [0]:
new_file_paths = [
    file_path
    for file_path in available_file_paths
    if file_path not in processed_file_paths
]

print(f"Available files : {len(available_file_paths)}")
print(f"Previously loaded: {len(processed_file_paths)}")
print(f"New files       : {len(new_file_paths)}")

In [0]:
if not new_file_paths:
    dbutils.notebook.exit(
        f"No new files found for dataset={dataset_name}"
    )

In [0]:
from datetime import datetime

pipeline_name = "retail_reference_data_ingestion"
notebook_name = "02_ingest_reference_bronze"
start_timestamp = datetime.now()

records_read = 0
records_written = 0
records_rejected = 0

In [0]:
from pyspark.sql import Row

def write_pipeline_audit(
    run_status,
    records_read=0,
    records_written=0,
    records_rejected=0,
    error_message=None
):
    audit_record = [
        Row(
            pipeline_run_id=pipeline_run_id,
            pipeline_name=pipeline_name,
            notebook_name=notebook_name,
            source_system=source_system,
            source_file_name=",".join(
                path.split("/")[-1]
                for path in new_file_paths
            ),
            source_file_path=",".join(new_file_paths),
            layer="bronze",
            load_type="INCREMENTAL",
            run_status=run_status,
            records_read=records_read,
            records_written=records_written,
            records_rejected=records_rejected,
            start_timestamp=start_timestamp,
            end_timestamp=datetime.now(),
            error_message=error_message,
            created_timestamp=datetime.now()
        )
    ]

    (
        spark.createDataFrame(audit_record)
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(audit_table)
    )

In [0]:
raw_df = (
    spark.read
    .format("csv")
    .schema(raw_schema)
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option(
        "columnNameOfCorruptRecord",
        "_corrupt_record"
    )
    .option("quote", '"')
    .option("escape", '"')
    .load(new_file_paths)
)

In [0]:
raw_df.printSchema()
display(raw_df.limit(20))

records_read = raw_df.count()

print(f"Records read: {records_read}")

In [0]:
bronze_df = (
    raw_df
    .withColumn(
        "_source_file_path",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "_source_file_name",
        F.regexp_extract(
            F.col("_metadata.file_path"),
            r"([^/]+$)",
            1
        )
    )
    .withColumn(
        "_source_file_size",
        F.col("_metadata.file_size")
    )
    .withColumn(
        "_source_file_modification_time",
        F.col("_metadata.file_modification_time")
    )
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_ingestion_date",
        F.current_date()
    )
    .withColumn(
        "_source_load_date",
        F.when(
            F.lit(load_date) != "",
            F.to_date(F.lit(load_date))
        ).otherwise(F.current_date())
    )
    .withColumn(
        "_pipeline_run_id",
        F.lit(pipeline_run_id)
    )
    .withColumn(
        "_source_system",
        F.lit(source_system)
    )
)

In [0]:
business_columns = [
    field.name
    for field in raw_schema.fields
    if field.name != "_corrupt_record"
]

print(business_columns)

In [0]:
bronze_df = bronze_df.withColumn(
    "_record_hash",
    F.sha2(
        F.concat_ws(
            "||",
            *[
                F.coalesce(
                    F.col(column_name),
                    F.lit("")
                )
                for column_name in business_columns
            ]
        ),
        256
    )
)

In [0]:
malformed_df = bronze_df.filter(
    F.col("_corrupt_record").isNotNull()
)

valid_structure_df = bronze_df.filter(
    F.col("_corrupt_record").isNull()
)

records_rejected = malformed_df.count()
records_written = valid_structure_df.count()

print(f"Records read       : {records_read}")
print(f"Structurally valid : {records_written}")
print(f"Malformed records  : {records_rejected}")

In [0]:
if records_rejected > 0:
    (
        malformed_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(quarantine_table)
    )

    print(
        f"Malformed records written to {quarantine_table}"
    )
else:
    print("No malformed physical CSV records found")

In [0]:
(
    valid_structure_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(bronze_table)
)

print(
    f"{records_written} records written to {bronze_table}"
)

In [0]:
file_metadata_by_path = {
    file.path: file
    for file in source_files
}

In [0]:
processed_file_records = []

for file_path in new_file_paths:
    file_info = file_metadata_by_path[file_path]

    processed_file_records.append(
        Row(
            source_file_path=file_path,
            source_file_name=file_info.name,
            source_system=source_system,
            target_table=bronze_table,
            file_size_bytes=file_info.size,
            file_modification_time=datetime.fromtimestamp(
                file_info.modificationTime / 1000
            ),
            pipeline_run_id=pipeline_run_id,
            processed_timestamp=datetime.now(),
            processing_status="SUCCESS"
        )
    )

In [0]:
(
    spark.createDataFrame(processed_file_records)
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(processed_files_table)
)

In [0]:
write_pipeline_audit(
    run_status="SUCCESS",
    records_read=records_read,
    records_written=records_written,
    records_rejected=records_rejected,
    error_message=""
)

In [0]:
print({
    "pipeline_run_id": pipeline_run_id,
    "dataset": dataset_name,
    "status": "SUCCESS",
    "records_read": records_read,
    "records_written": records_written,
    "records_rejected": records_rejected,
    "target_table": bronze_table
})

In [0]:
bronze_tables = [
    "products",
    "customers",
    "stores",
    "inventory",
    "promotions"
]

for table_name in bronze_tables:
    full_table_name = (
        f"{catalog_name}.bronze.{table_name}"
    )

    row_count = spark.table(full_table_name).count()

    print(
        f"{full_table_name}: {row_count} records"
    )

In [0]:
display(
    spark.table(audit_table)
    .filter(
        F.col("notebook_name")
        == "02_ingest_reference_bronze"
    )
    .orderBy(
        F.col("created_timestamp").desc()
    )
)

In [0]:
try:
    # Read source
    # Add metadata
    # Split malformed records
    # Write quarantine
    # Write Bronze
    # Register file
    # Write success audit

    pass

except Exception as exc:
    error_message = str(exc)[:4000]

    write_pipeline_audit(
        run_status="FAILED",
        records_read=records_read,
        records_written=records_written,
        records_rejected=records_rejected,
        error_message=error_message
    )

    raise

In [0]:
display(
    spark.table(
        "retail_lakehouse.audit.processed_files"
    )
    .orderBy(F.col("processed_timestamp").desc())
)